# Breaking the Blend - automated decomposition of IFS data (example: NGC 5728)

This notebook is an **automated implementation of the Breaking the Blend method** (Veenema et al. 2026):
separating, spaxel by spaxel, the emission of two physically distinct components - here a circumnuclear
star-forming disc/ring and an AGN-driven biconical outflow - using many emission lines at once.

The example loads JWST NIRSpec + MIRI/MRS data of **NGC 5728**, but every choice (target, cubes, lines,
fit settings, model parameters, which component each line traces) is set in a few clearly marked cells
and can be changed for any other galaxy or set of lines.

| Section | What happens | Module |
|---|---|---|
| 0 | Settings for this analysis (name, data folder, redshift, run switches) | `project` |
| 1 | Find & load cubes, check the AGN position | `cubes` |
| 2 | Choose the emission lines and their fit settings | `lines` |
| 3 | **Step 1** - flux, single-/double-Gaussian and moment maps, dispersion of each component | `linefit`, `dispersion` |
| 4 | **Step 2** - disc / bicone / combined velocity models (fitted, fixed, or imported from other codes) | `models`, `kinfit`, `external` |
| 5 | **Step 3** - constrained decomposition -> disc fraction $f_{\rm disc}$ per line | `decompose` |
| 6 | **Step 4** - weighted stack of all lines -> AGN outflow fraction map | `stacking` |
| 7 | **Step 5** - AGN- and star-formation-dominated spectra | `extraction` |

Every slow step is **saved** in `results/<GALAXY>/` and re-loaded next time; if its settings change it is recomputed
automatically (`RERUN_*` forces a re-run). Figures go to `results/<GALAXY>/figures/`, settings to `config.json`.

**Data.** Download the public MAST cubes into `data/NGC5728/` first (see `data/NGC5728/README.md`). MAST cubes are
reduced with the current JWST pipeline, so results will be close to, but not identical with, the published ones.

**About the example settings.** An automated analysis cannot make every per-line choice that a careful by-hand
analysis does. To reproduce the published NGC 5728 maps closely, this example therefore adopts several settings
of the original analysis (per-line masks, AGN positions, fit windows and the per-line best-fit model parameters
and dispersions). Each is marked *"original NGC 5728 analysis"* and can be switched to the automated / newly
fitted values. Results obtained with different choices will differ in detail - see the notes at the end.

## 0. Settings

In [ ]:
# =====================================================================
# 0. ANALYSIS SETTINGS - edit this cell for a new galaxy / analysis
# =====================================================================
GALAXY       = "NGC5728"            # analysis name -> results are saved in RESULTS_DIR/GALAXY/
DATA_DIR     = "../data/NGC5728"    # folder with the *_s3d.fits(.gz) cubes (searched recursively) - see data/NGC5728/README.md
RESULTS_DIR  = "results"            # parent folder for all analyses
Z            = 0.00935              # redshift
DISTANCE_MPC = None                 # e.g. 39 to fix the physical scale; None = from the redshift

# Re-run switches: False -> load the saved result if it exists (and its settings are unchanged)
RERUN_LINE_FITS     = False
RERUN_KINEMATICS    = False
RERUN_DECOMPOSITION = False

# Kinematic fitting (Step 2). The paper used nsteps=150000, burnin=50000 (slow).
# method="least_squares" is much faster for exploring. sigma_v sets the velocity uncertainty per spaxel
# in the likelihood: "residual" (scatter about the best fit; realistic errors) or "std" (scatter of the map).
MCMC = dict(method="emcee", nwalkers=32, nsteps=20000, burnin=5000, sigma_v="residual")

# Figure output: "png", "pdf", "both" or "none"  (saved in RESULTS_DIR/GALAXY/figures/)
SAVE_FIGURES = "both"
FIGURE_DPI   = 200

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
try:
    import breaking_the_blend as btb
except ModuleNotFoundError:   # not installed (pip install -e .): use the copy in this repository
    import sys, os
    sys.path.insert(0, os.path.abspath(".."))
    import breaking_the_blend as btb
from breaking_the_blend import plotting as bplt     # also sets the publication plotting style

proj = btb.Project(GALAXY, data_dir=DATA_DIR, z=Z, distance_mpc=DISTANCE_MPC, results_dir=RESULTS_DIR,
                   save_figures=SAVE_FIGURES, figure_dpi=FIGURE_DPI)

## 1. Data: find the cubes and locate the AGN

**Getting the data.** The cubes are public on MAST (JWST programmes **1670**, MIRI/MRS, and **5017**, NIRSpec IFU).
Download them into `data/NGC5728/` with the script (from the repository root):
`python scripts/download_mast_cubes.py --target "NGC 5728" --programs 1670 5017 --out data/NGC5728`,
or via the MAST portal - full instructions in `data/NGC5728/README.md`.

`find_cubes` searches `DATA_DIR` (and all sub-folders, e.g. a raw MAST download) for `*s3d.fits*` files and
names each by its band (`ch3-short`, `g395h-f290lp`, ...). Use `exclude=[...]` to skip files, or
`proj.set_cube_files({...})` to give them by hand. Cubes are converted to **Jy/pixel** and de-redshifted.

The AGN position of each cube defaults to the (centroided) continuum peak. All maps are then expressed as
**offsets from the AGN in parsec**, so bands with different pixel sizes line up. `AGN_OVERRIDES` sets positions
by hand (here: the positions adopted in the original NGC 5728 analysis).

The pixel size is read from each cube header (`PIXEL_SCALE_OVERRIDES` can replace it). The overview figure marks the
continuum peak of each cube; the printed table lists the adopted AGN positions.

In [ ]:
files = proj.find_cubes(exclude=["*bkg*"])

In [ ]:
# AGN pixel positions (x, y) adopted in the original NGC 5728 analysis. These only apply to the cubes used in that
# analysis (GATOS reduction): for cubes downloaded from MAST keep USE_ORIGINAL_AGN_XY = False, which uses the
# continuum peak of every cube (pixel grids differ between reductions).
USE_ORIGINAL_AGN_XY = False
ORIGINAL_AGN_XY = {"ch1-short": (39.8, 39.5), "ch1-long": (39.8, 38.5), "ch2-short": (36.8, 32.0),
                   "ch2-medium": (36.7, 35.5), "ch2-long": (37.7, 34.5), "ch3-short": (34.0, 37.0),
                   "ch3-medium": (35.0, 37.5), "ch3-long": (34.7, 36.5), "ch4-short": (24.5, 22.5),
                   "ch4-long": (24.8, 22.1), "g395h-f290lp": (49.0, 43.5)}
AGN_OVERRIDES = ORIGINAL_AGN_XY if USE_ORIGINAL_AGN_XY else {}

PIXEL_SCALE_OVERRIDES = {}          # {"cube": arcsec} only if a header pixel size is wrong

cubes = proj.load_cubes(agn_overrides=AGN_OVERRIDES, pixel_scale_overrides=PIXEL_SCALE_OVERRIDES)
fig = bplt.plot_cubes_overview(cubes)
proj.savefig(fig, "cubes_overview")

## 2. Emission lines

Each line is a `btb.LineSpec` (see `btb.LINE_CATALOGUE`). Pick lines by name, then change any setting in
`LINE_SETTINGS`:

* `cube` - cube to measure the line in (left out -> chosen automatically);
* `line_region` - rest-frame range ($\mu$m) holding the line: excluded from the continuum fit and integrated;
* `fit_window` - range cut out of the cube (default `line_region` $\pm$ 0.02 $\mu$m);
* `p0_amp_frac`, `p0_dv_kms`, `p0_sigma_kms` - initial (narrow, broad) amplitude fraction, velocity offset and
  dispersion of the double-Gaussian fit;
* `mask_percentile` - spaxels fainter than this flux percentile are masked.

The values below are those of the original NGC 5728 analysis (initial guesses converted from microns to km/s).
A line not in the catalogue can be added with `btb.LineSpec("name", "label", wavelength, IP)`.

In [ ]:
LINES = ["FeII", "ArII", "NeII", "SIII", "ArIII", "SIV", "NeIII", "OIV", "ArVI", "MgIV", "NeV", "NeVI"]

LINE_SETTINGS = {   # original NGC 5728 analysis
    "FeII":  dict(cube="ch1-short",  line_region=(5.320, 5.355),  mask_percentile=55,
                  p0_amp_frac=(0.67, 0.33), p0_dv_kms=(-2.8, 1.1),  p0_sigma_kms=(112, 225)),
    "ArII":  dict(cube="ch1-long",   line_region=(6.975, 7.000),  mask_percentile=50,
                  p0_amp_frac=(0.77, 0.50), p0_dv_kms=(-21.5, 4.3), p0_sigma_kms=(86, 215)),
    "NeII":  dict(cube="ch3-short",  line_region=(12.800, 12.830), mask_percentile=70,
                  p0_amp_frac=(0.50, 0.33), p0_dv_kms=(-23.4, 2.3), p0_sigma_kms=(105, 164)),
    "SIII":  dict(cube="ch4-short",  line_region=(18.650, 18.780), mask_percentile=55,
                  p0_amp_frac=(0.71, 0.40), p0_dv_kms=(-2.4, 3.2),  p0_sigma_kms=(112, 152)),
    "ArIII": dict(cube="ch2-medium", line_region=(8.980, 9.015),  mask_percentile=55,
                  p0_amp_frac=(0.71, 0.43), p0_dv_kms=(-1.7, 1.7),  p0_sigma_kms=(100, 200)),
    "SIV":   dict(cube="ch2-long",   line_region=(10.485, 10.535), mask_percentile=55,
                  p0_amp_frac=(0.67, 0.33), p0_dv_kms=(0.0, 28.5),  p0_sigma_kms=(86, 257)),
    "NeIII": dict(cube="ch3-long",   line_region=(15.520, 15.600), mask_percentile=60,
                  p0_amp_frac=(0.77, 0.33), p0_dv_kms=(-3.9, 3.9),  p0_sigma_kms=(106, 173)),
    "OIV":   dict(cube="ch4-long",   line_region=(25.830, 25.970), mask_percentile=55,
                  p0_amp_frac=(0.67, 0.33), p0_dv_kms=(0.0, 1.2),   p0_sigma_kms=(81, 127)),
    "ArVI":  dict(cube="g395h-f290lp", line_region=(4.524, 4.537), mask_percentile=55,
                  p0_amp_frac=(0.33, 0.50), p0_dv_kms=(-3.3, 3.3),  p0_sigma_kms=(66, 165)),
    "MgIV":  dict(cube="g395h-f290lp", line_region=(4.481, 4.492), mask_percentile=55,
                  p0_amp_frac=(0.33, 0.67), p0_dv_kms=(-0.7, 0.1),  p0_sigma_kms=(134, 267)),
    "NeV":   dict(cube="ch3-medium", line_region=(14.295, 14.345), fit_window=(14.265, 14.350),  # avoid [Cl II]
                  mask_percentile=50),
    "NeVI":  dict(cube="ch2-short",  line_region=(7.645, 7.670),  mask_percentile=50,
                  p0_amp_frac=(0.33, 0.67), p0_dv_kms=(-0.4, 0.0),  p0_sigma_kms=(78, 176)),
}

lines = btb.get_lines(LINES)                                         # ordered by IP
lines = {n: l.update(**LINE_SETTINGS.get(n, {})) for n, l in lines.items()}
lines = btb.assign_cubes(lines, cubes)                               # checks / fills the cube of each line
proj.save_config(lines=lines)

## 3. Step 1 - measure every line in every spaxel

For each line and spaxel `fit_lines` subtracts a polynomial continuum, integrates the line flux, masks spaxels
below `mask_percentile`, fits a **single** and a **double** Gaussian (the narrow component is always the one with
the smaller $\sigma$) and computes moment maps. Fit options:

* `fitter` - `"trf"` (bounded, default), `"lm"` (unbounded Levenberg-Marquardt) or `"capfit"` (unbounded CapFit,
  as in the original analysis; `pip install capfit`, falls back to `"lm"` if missing);
* `weighting` - `"none"`, `"err"` (cube ERR extension) or `"sqrt"` ($\sqrt{|f|}$ uncertainties, as in the original);
* `p0_center` - start the Gaussians at the moment-1 velocity (`"moment"`) or at systemic (`"systemic"`, original);
* `percentile_over` - flux percentile over the whole array (`"all"`, original) or inside the field of view (`"fov"`).

**These choices matter for the double-Gaussian dispersions** (next sub-section): the `"sqrt"` weighting gives the faint
line wings much more weight, which widens the broad component. No correction for the instrumental line-spread
function is applied (as in the paper), so that is not a source of difference. The options below reproduce the
original analysis; `fitter="trf", weighting="none", p0_center="moment", percentile_over="fov"` is a more
conservative automated choice.

In [ ]:
FIT_OPTIONS = dict(fitter="capfit", weighting="sqrt", p0_center="systemic", percentile_over="all", dilate=2)

line_maps = proj.checkpoint("step1_line_maps", btb.fit_lines, cubes, lines, **FIT_OPTIONS,
                            rerun=RERUN_LINE_FITS, settings=dict(lines=lines, **FIT_OPTIONS))

**Check the fits first.** If the double-Gaussian fits in a few spaxels look wrong, change `p0_*` / `line_region`
for that line in Section 2 (it is re-fitted automatically).

In [ ]:
fig = bplt.plot_spaxel_fits(line_maps["NeII"], n=6);  proj.savefig(fig, "spaxel_fits_NeII")

### Flux and single-Gaussian maps (paper Figs. S1, S2)

In [ ]:
fig = bplt.plot_line_maps(line_maps, "flux");      proj.savefig(fig, "S1_flux_maps")
fig = bplt.plot_line_maps(line_maps, "velocity");  proj.savefig(fig, "S2_velocity_maps")
fig = bplt.plot_line_maps(line_maps, "sigma");     proj.savefig(fig, "single_gaussian_dispersion_maps")

These maps, ordered by IP, show the transition from disc-dominated (low IP) to outflow-dominated (high IP)
emission - they are where you decide **which lines trace which component** (used in Step 2).

### Double-Gaussian maps (narrow | broad)

In [ ]:
fig = bplt.plot_line_quantities(line_maps, ("v_n", "v_b"));          proj.savefig(fig, "double_gaussian_velocity_maps")
fig = bplt.plot_line_quantities(line_maps, ("sigma_n", "sigma_b"));  proj.savefig(fig, "double_gaussian_dispersion_maps")

### Moment maps (0: flux, 1: velocity, 2: dispersion)

In [ ]:
fig = bplt.plot_line_quantities(line_maps, ("mom0", "mom1", "mom2")); proj.savefig(fig, "moment_maps")

### Velocity dispersion of each component (paper Fig. S3)

Enforcing $\sigma_{\rm outflow} > \sigma_{\rm disc}$ in each spaxel, the medians of the two distributions are the
candidate fixed dispersions for Step 3. `SIGMA_OVERRIDES` replaces measured values (the published Table 1 values
are given for comparison).

In [ ]:
TABLE1_SIGMAS = {"FeII": (78, 203), "ArII": (59, 159), "NeII": (55, 134), "SIII": (72, 175),
                 "ArIII": (67, 163), "SIV": (80, 193), "NeIII": (72, 194), "OIV": (81, 176),
                 "ArVI": (51, 139), "MgIV": (71, 197)}
SIGMA_OVERRIDES = {}                 # e.g. TABLE1_SIGMAS

sigmas_measured = btb.dispersion_medians(line_maps, sigma_range=(15, 600), overrides=SIGMA_OVERRIDES)
fig = bplt.plot_dispersion_histograms(line_maps, sigmas_measured)
proj.savefig(fig, "S3_dispersion_histograms")
proj.save_config(sigmas_measured=sigmas_measured)

## 4. Step 2 - kinematic models

Two components, evaluated at offsets from the AGN in **pc** (angles in degrees, PA east of north):

* **thin rotating disc**: $v = v_{\rm sys} + v_{\rm disc}\tanh(R/R_{\rm disc})\sin i_{\rm disc}\cos\theta$
  - parameters `v_disc`, `R_disc`, `inc_disc`, `pa_disc` (receding side);
* **hollow bicone**: radial velocity rising linearly to `v_out` at `R_out`, falling to 0 at 2`R_out`, inside opening
  angles `theta_in`-`theta_out` around an axis with `inc_out`, `pa_out` (redshifted cone);
* **combined**: $v = \alpha\,v_{\rm disc} + (1-\alpha)\,v_{\rm outflow}$ (`alpha`); plus `v_sys`.

### How to fix or fit each parameter

All parameters live in **one dict**, `MODEL_PARAMS`. For each entry:

* a **number** -> the parameter is **fixed** to that value;
* a **tuple `(lo, hi)`** -> the parameter is **fitted**, with a uniform prior between `lo` and `hi`.

Any combination is allowed - e.g. to fit the disc inclination replace `43.0` by `(20, 70)`; to fix the outflow
speed replace `(100, 1500)` by `700`. The same dict is used for every fit below (a disc-only fit simply ignores
the outflow entries). `init` (optional) narrows where the MCMC walkers start without changing the priors.
For NGC 5728 the geometry is fixed to literature values because the JWST field covers only part of the ring.

In [ ]:
MODEL_PARAMS = {
    # ---- systemic velocity ------------------------------------------------------------------
    "v_sys":     0.0,           # km/s   (0 if the redshift is right)
    # ---- disc ----------------------------------------------------------------------------------
    "inc_disc":  43.0,          # deg    Shimizu et al. (2019)       e.g. (20, 70) to fit
    "pa_disc":   194.0,         # deg E of N, Shimizu et al. (2019) e.g. (150, 240) to fit
    "v_disc":    (50, 800),     # km/s   maximum rotation speed
    "R_disc":    (20, 1500),    # pc     turnover radius of the tanh rotation curve
    # ---- outflow (bicone) ----------------------------------------------------------------------
    "inc_out":   48.0,          # deg    Shimizu et al. (2019)
    "pa_out":    110.0,         # deg E of N
    "v_out":     (100, 1500),   # km/s   peak radial outflow speed
    "R_out":     (50, 3000),    # pc     radius of the peak
    "theta_in":  (0, 60),       # deg    inner opening angle
    "theta_out": (30, 120),     # deg    outer opening angle
    # ---- combined model -----------------------------------------------------------------------
    "alpha":     (0, 1),        # disc weight of the velocity
}

DISC_LINES      = ["FeII", "ArII", "NeII"]                 # disc-dominated (low IP)
OUTFLOW_LINES   = ["OIV", "ArVI", "MgIV", "NeV", "NeVI"]   # outflow-dominated (high IP)
DECOMPOSE_LINES = ["FeII", "ArII", "NeII", "SIII", "ArIII", "SIV", "NeIII", "OIV", "ArVI", "MgIV"]
# [Ne V] and [Ne VI] show no disc component in NGC 5728 -> they only constrain the outflow

RUN_KINEMATIC_FITS = True      # False -> skip the fits (Step 3 can still use fixed values)

The fitting sequence: (1) disc-only fits to disc-dominated lines, (2) outflow-only fits to outflow-dominated lines,
(3) combined fits to all decomposable lines started near (1) and (2), (4) averages of IP-independent parameters and
linear trends with IP of those that vary ($R_{\rm out}$, $\alpha$).

In [ ]:
if RUN_KINEMATIC_FITS:
    disc_fits = proj.checkpoint("step2a_disc_fits", btb.fit_many, line_maps, DISC_LINES, "disc",
                                params=MODEL_PARAMS, **MCMC, rerun=RERUN_KINEMATICS,
                                settings=dict(lines=DISC_LINES, params=MODEL_PARAMS, **MCMC))
    fig = bplt.plot_kinematic_fit(disc_fits["NeII"], line_maps["NeII"]);  proj.savefig(fig, "disc_fit_NeII")

In [ ]:
if RUN_KINEMATIC_FITS:
    outflow_fits = proj.checkpoint("step2b_outflow_fits", btb.fit_many, line_maps, OUTFLOW_LINES, "outflow",
                                   params=MODEL_PARAMS, **MCMC, rerun=RERUN_KINEMATICS,
                                   settings=dict(lines=OUTFLOW_LINES, params=MODEL_PARAMS, **MCMC))
    fig = bplt.plot_kinematic_fit(outflow_fits["NeV"], line_maps["NeV"]);  proj.savefig(fig, "outflow_fit_NeV")

In [ ]:
if RUN_KINEMATIC_FITS:
    start = {**btb.mean_params(disc_fits, [k for k in ("v_disc", "R_disc") if not np.isscalar(MODEL_PARAMS[k])]),
             **btb.mean_params(outflow_fits, [k for k in ("v_out", "R_out", "theta_in", "theta_out")
                                              if not np.isscalar(MODEL_PARAMS[k])])}
    init = {k: (0.7 * v, 1.3 * v + 1) for k, v in start.items()}
    combined_fits = proj.checkpoint("step2c_combined_fits", btb.fit_many, line_maps, DECOMPOSE_LINES,
                                    "combined", params=MODEL_PARAMS, init=init, **MCMC, rerun=RERUN_KINEMATICS,
                                    settings=dict(lines=DECOMPOSE_LINES, params=MODEL_PARAMS, init=init, **MCMC))
    btb.fits_table(combined_fits)
    fig = bplt.plot_kinematic_fit(combined_fits["SIII"], line_maps["SIII"]); proj.savefig(fig, "S4_SIII_kinematic_fit")

Parameters against IP. Boxes show the 25-75th percentiles of each posterior, whiskers the 16-84th, the black line
the median (`style="errorbar"` for error bars).

In [ ]:
if RUN_KINEMATIC_FITS:
    fig = bplt.plot_params_vs_ip(combined_fits, ["R_out", "alpha", "v_disc", "v_out"], style="box")
    proj.savefig(fig, "params_vs_IP")
    FIT_MEANS = btb.mean_params(combined_fits, ["v_disc", "R_disc", "v_out", "theta_in", "theta_out"])
    R_OUT_TREND = btb.trend_with_ip(combined_fits, "R_out")
    ALPHA_TREND = btb.trend_with_ip(combined_fits, "alpha", max_ip=81)

### Optional: velocity / dispersion maps from other codes (3DBarolo, MOKA3D, ...)

Instead of the built-in models, any component can use a map produced elsewhere, e.g. a disc velocity (and
dispersion) field from **3DBarolo** or an outflow velocity field from **MOKA3D**. Give the AGN pixel position *in that
map*, its pixel size, and the systemic velocity to subtract if it holds absolute velocities; the map is then
re-projected onto every line's grid like the models. Where an imported map is undefined the built-in model (or
fixed dispersion) is used. Keys: `"v_disc"`, `"v_out"`, `"sigma_disc"`, `"sigma_out"`. Off by default.

In [ ]:
USE_EXTERNAL_MAPS = False
EXTERNAL = {}

if USE_EXTERNAL_MAPS:
    pc_scale = next(iter(cubes.values())).pc_per_arcsec           # pc per arcsec
    EXTERNAL["v_disc"] = btb.load_external_map("path/to/3dbarolo_model_velocity.fits",
                                               agn_xy=(40.0, 40.0),      # AGN / kinematic centre in that map
                                               pixel_scale_arcsec=0.2, pc_per_arcsec=pc_scale,
                                               subtract=0.0)             # systemic velocity if absolute
    # EXTERNAL["sigma_disc"] = btb.load_external_map("path/to/3dbarolo_model_dispersion.fits", ...)
    # EXTERNAL["v_out"]      = btb.load_external_map("path/to/moka3d_velocity_map.fits", ...)

    # check the alignment against an observed velocity map
    lm = line_maps["NeII"]
    fig = bplt.plot_velocity_panels({"observed": lm.velocity_map(),
                                     "imported disc": np.where(lm.good, EXTERNAL["v_disc"].on_grid(lm), np.nan)},
                                    lm.extent_pc(), vlim=400)
    proj.savefig(fig, "external_map_check")

## 5. Step 3 - Breaking the blend: constrained decomposition per spaxel

In each good spaxel every decomposable line is fitted with two Gaussians whose **centroids are fixed** to the disc
and outflow velocities and whose **dispersions are fixed**, so only the two amplitudes are free (non-negative
least squares): $f_{\rm disc} = A_{\rm disc}\sigma_{\rm disc} / (A_{\rm disc}\sigma_{\rm disc} + A_{\rm out}\sigma_{\rm out})$.

### Which model parameters and dispersions to use

`DECOMP_PARAMS` selects the model parameters of each line:

* `"fixed_per_line"` - `FIXED_LINE_PARAMS`: one set per line (here the per-line best fits of the original NGC 5728
  analysis; radii in pixels of each line's cube, converted to pc below);
* `"fixed_common"` - `FIXED_COMMON` for all lines plus $R_{\rm out}(IP)$ from `FIXED_R_OUT`
  (here the averaged values and trend of the original analysis);
* `"fit_mean"` - the geometry plus the averages of the combined fits of Section 4, and the fitted $R_{\rm out}(IP)$;
* `"fit_per_line"` - each line's own combined fit from Section 4.

`DECOMP_SIGMAS` selects the dispersions: `"fixed"` (`FIXED_SIGMAS`, here those of the original analysis) or
`"measured"` (the Step 1 medians).

In [ ]:
DECOMP_PARAMS = "fixed_per_line"    # "fixed_per_line" | "fixed_common" | "fit_mean" | "fit_per_line"
DECOMP_SIGMAS = "fixed"             # "fixed" | "measured"

GEOMETRY = {k: MODEL_PARAMS[k] for k in ("v_sys", "inc_disc", "pa_disc", "inc_out", "pa_out")}

# --- per-line values (original NGC 5728 analysis) ---------------------------------------------------------
#                v_disc  R_disc  v_out  R_out   theta_in theta_out     (km/s, PIXELS, km/s, PIXELS, deg, deg)
FIXED_LINE_PARAMS_PIX = {
    "FeII":  (457.6,  7.09, 590.0, 13.13, 21.0,  80.0),
    "ArII":  (420.9,  7.14, 483.6, 14.83, 21.0,  80.0),
    "NeII":  (399.8,  6.93, 528.0,  9.84, 22.3,  90.0),
    "SIII":  (393.0,  4.98, 588.0,  5.51, 21.0,  90.0),
    "ArIII": (444.0, 10.22, 752.0, 11.11, 21.0,  70.0),
    "SIV":   (438.0, 18.70, 697.0, 10.64, 21.0, 102.0),
    "NeIII": (381.0, 11.31, 567.0,  9.39, 21.0,  80.0),
    "OIV":   (337.0,  6.35, 612.0,  6.37, 21.0, 101.0),
    "ArVI":  (294.5,  6.83, 665.0, 15.07, 21.0,  77.0),
    "MgIV":  (202.9, 14.43, 855.0, 11.30, 21.0,  64.0),
}
# --- common values (averages of the original analysis) ----------------------------------------------------
FIXED_COMMON = dict(v_disc=400.0, R_disc=290.0, v_out=700.0, theta_in=21.0, theta_out=80.0)
FIXED_R_OUT  = (3.6, 535.0)                        # R_out = 3.6 * IP + 535 pc
# --- dispersions (sigma_disc, sigma_outflow) [km/s] used in the original decomposition --------------------
FIXED_SIGMAS = {"FeII": (78, 203), "ArII": (59, 159), "NeII": (81, 176), "SIII": (72, 175),
                "ArIII": (67, 163), "SIV": (80, 193), "NeIII": (72, 176), "OIV": (81, 176),
                "ArVI": (51, 139), "MgIV": (71, 197)}
# In the original notebook the flux fractions of these two lines used the Table 1 dispersions while the
# amplitude fit used FIXED_SIGMAS. Kept here to reproduce the published maps; set to {} for the
# self-consistent calculation.
FRACTION_SIGMA_OVERRIDES = {"NeII": (55, 134), "NeIII": (72, 194)}
# Spectra used in the original decomposition: wider windows with a linear continuum. Set to {} to use the
# Step 1 spectra instead.            window (rest um)   line-free outside (rest um)
DECOMP_RECUT = {n: dict(window=w, line_region=lr, cont_order=1) for n, (w, lr) in {
    "FeII": ((5.25, 5.45), (5.33, 5.35)),       "ArII": ((6.90, 7.08), (6.976, 6.993)),
    "NeII": ((12.72, 12.92), (12.785, 12.835)), "SIII": ((18.62, 18.82), (18.680, 18.740)),
    "ArIII": ((8.94, 9.07), (8.975, 9.005)),    "SIV": ((10.43, 10.59), (10.49, 10.53)),
    "NeIII": ((15.45, 15.65), (15.54, 15.59)),  "OIV": ((25.80, 25.98), (25.84, 25.94)),
    "ArVI": ((4.49, 4.59), (4.525, 4.535)),     "MgIV": ((4.42, 4.56), (4.47, 4.50))}.items()}

In [ ]:
dmaps = {n: line_maps[n] for n in DECOMPOSE_LINES}
if DECOMP_PARAMS == "fixed_per_line":
    per_line = {n: dict(v_disc=vd, R_disc=rd * dmaps[n].pixel_scale_pc, v_out=vo,
                        R_out=ro * dmaps[n].pixel_scale_pc, theta_in=ti, theta_out=to)
                for n, (vd, rd, vo, ro, ti, to) in FIXED_LINE_PARAMS_PIX.items() if n in dmaps}
    line_parameters = btb.line_params(dmaps, GEOMETRY, per_line=per_line)
elif DECOMP_PARAMS == "fixed_common":
    a, b = FIXED_R_OUT
    line_parameters = btb.line_params(dmaps, {**GEOMETRY, **FIXED_COMMON},
                                      ip_relations={"R_out": lambda ip: a * ip + b})
elif DECOMP_PARAMS == "fit_mean":
    a, b = R_OUT_TREND
    line_parameters = btb.line_params(dmaps, {**GEOMETRY, **FIT_MEANS},
                                      ip_relations={"R_out": lambda ip: a * ip + b})
elif DECOMP_PARAMS == "fit_per_line":
    line_parameters = btb.line_params(dmaps, GEOMETRY, fits=combined_fits)
else:
    raise ValueError(DECOMP_PARAMS)

if DECOMP_SIGMAS == "fixed":
    sigmas = {n: {"disc": FIXED_SIGMAS[n][0], "outflow": FIXED_SIGMAS[n][1]} for n in DECOMPOSE_LINES}
else:
    sigmas = sigmas_measured

proj.save_config(decomp_params_source=DECOMP_PARAMS, decomp_sigmas_source=DECOMP_SIGMAS,
                 decomposition_params=line_parameters, decomposition_sigmas=sigmas)

In [ ]:
DECOMP_OPTIONS = dict(names=DECOMPOSE_LINES, sigma_mode="fixed", recut=DECOMP_RECUT,
                      sigma_fraction=FRACTION_SIGMA_OVERRIDES)

decomps = proj.checkpoint("step3_decompositions", btb.decompose_lines, line_maps, line_parameters, sigmas,
                          cubes=cubes, external=EXTERNAL or None, **DECOMP_OPTIONS, rerun=RERUN_DECOMPOSITION,
                          settings=dict(params=line_parameters, sigmas=sigmas, external=sorted(EXTERNAL),
                                        **DECOMP_OPTIONS))

Disc-fraction map of every line (paper Fig. S6):

In [ ]:
fig = bplt.plot_fraction_maps(decomps, "f_disc");  proj.savefig(fig, "S6_disc_fraction_maps")
# fig = bplt.plot_residual_maps(decomps)          # coherent residuals would point to a missing component
# fig = bplt.plot_decomposition_spaxels(decomps["NeIII"], line_maps["NeIII"])   # individual spaxel fits

## 6. Step 4 - stack all lines into one map (paper Fig. 2)

Each line's $f_{\rm disc}$ map is interpolated onto a common parsec grid centred on the AGN and averaged with a weight
$w = 1 - 2|\langle f_{\rm disc}\rangle - 0.5|$ favouring lines that trace both components. Only points covered by at
least `min_lines` lines are kept.

In [ ]:
STACK_OPTIONS = dict(scheme="linear", combine="mean", min_lines=2, grid_halfsize_pc=1500)

stack = btb.stack_fraction_maps(decomps, **STACK_OPTIONS)
print(f"line-to-line scatter in f_disc: {btb.line_to_line_scatter(stack):.2f}")

fig = bplt.plot_stacked_map(stack, which="f_out", lim_pc=1500, scalebar_pc=570)
proj.savefig(fig, "Fig2_outflow_fraction")
proj.save(stack, "step4_stacked_map")
proj.export_stacked_map(stack)

**Robustness** (paper Sect. 3.4): the map should not depend on the weighting or on any single line.

In [ ]:
alternatives = {
    "linear weights":   stack,
    "equal weights":    btb.stack_fraction_maps(decomps, **{**STACK_OPTIONS, "scheme": "equal"}, verbose=False),
    "gaussian weights": btb.stack_fraction_maps(decomps, **{**STACK_OPTIONS, "scheme": "gaussian"}, verbose=False),
    "median":           btb.stack_fraction_maps(decomps, **{**STACK_OPTIONS, "combine": "median"}, verbose=False),
    "without [Fe II]":  btb.stack_fraction_maps(decomps, names=[n for n in DECOMPOSE_LINES if n != "FeII"],
                                                **STACK_OPTIONS, verbose=False),
}
btb.compare_stacks(alternatives, "linear weights")

fig, axes = plt.subplots(1, len(alternatives), figsize=(4.6 * len(alternatives), 4.0), constrained_layout=True)
for ax, (k, s) in zip(axes, alternatives.items()):
    bplt.plot_stacked_map(s, lim_pc=1500, ax=ax, title=k)
proj.savefig(fig, "stack_robustness")

## 7. Step 5 - AGN and star-forming spectra

The stacked map is projected onto **every** cube (NIRSpec + all MRS bands). Spaxels with $f_{\rm out} > 0.8$ give
the AGN-dominated spectrum and $f_{\rm out} < 0.2$ the star-formation-dominated one (`method="weighted"` instead
weights every spaxel by its fractions). The per-band spectra are stitched band by band, outwards from a reference band
(by default the one covering the largest sky area, so a small field of view does not set the normalisation). Channels
without data (e.g. the NIRSpec G395H detector gap near 4.1 $\mu$m) are left out, and bands with fewer than
`min_spaxels` selected spaxels are skipped for that component.

In [ ]:
spectra = btb.extract_all(cubes, stack, method="threshold", agn_min=0.8, sf_max=0.2, min_spaxels=3)
fig = bplt.plot_extraction_regions(spectra["ch3-short"], cubes["ch3-short"]);  proj.savefig(fig, "extraction_regions")

In [ ]:
w_agn, f_agn, _ = btb.stitch_spectra(spectra, "agn")
w_sf,  f_sf,  _ = btb.stitch_spectra(spectra, "sf")

fig = bplt.plot_component_spectra(w_agn, f_agn, w_sf, f_sf, smooth=2)
proj.savefig(fig, "component_spectra")
proj.save({"agn": (w_agn, f_agn), "sf": (w_sf, f_sf), "per_band": spectra}, "step5_spectra")
proj.export_spectra({"AGN": (w_agn, f_agn), "SF": (w_sf, f_sf)})

**What to expect.** The AGN-dominated spectrum should show a steeply rising hot-dust continuum, silicate absorption at
9.7 $\mu$m and strong high-ionisation lines ([Ne V], [Ne VI], [O IV]); the star-formation-dominated spectrum should show
strong PAH features (6.2, 7.7, 8.6, 11.3, 12.7, 17 $\mu$m), relatively stronger low-ionisation lines and a fainter, flatter
continuum. The SF spaxels are those with $f_{\rm out} <$ `sf_max` in the stacked map - typically the parts of the ring
furthest from the bicone - so in bands with a small field of view (NIRSpec, MRS channel 1) there may be few or none.

In [ ]:
proj.summary()

## Notes and caveats

### Automated versus by-hand analysis
This notebook applies the same recipe to every line. A careful by-hand analysis can do better for individual
lines, and the results here will differ in detail from a tailored analysis (including the published one) if
the settings differ. The quantities that matter most:

* **masks** (`mask_percentile`, `dilate`, `percentile_over`) - set the field of view of each line and thus of the stack;
* **AGN position of each cube** - misalignments of a pixel shift the per-line maps relative to each other;
* **pixel scales** - set the physical size of each band in the stacked map;
* **double-Gaussian fitting** (`fitter`, `weighting`, initial guesses) - sets the dispersion medians;
* **model parameters and dispersions per line** in the decomposition (`DECOMP_PARAMS`, `DECOMP_SIGMAS`);
* **fit windows and continuum** of the spectra that are decomposed (`DECOMP_RECUT`).

### Doing it line by line
Every step is a plain function, so any line can be treated individually and the result swapped into the
dictionaries before stacking, e.g.:

```python
lm = btb.fit_line(cubes["ch3-short"], lines["NeII"].update(line_region=(12.79, 12.84), mask_percentile=65),
                  fitter="capfit", weighting="sqrt")              # re-measure one line
bplt.plot_spaxel_fits(lm)                                          # inspect it
fit = btb.fit_kinematics(lm, "combined", {**MODEL_PARAMS, "theta_out": (60, 110)}, method="emcee")
d = btb.decompose_line(lm, fit.params, sigma_disc=55, sigma_out=134,
                       cube=cubes["ch3-short"], recut=dict(window=(12.72, 12.92), line_region=(12.785, 12.835)))
bplt.plot_decomposition_spaxels(d, lm)                             # check the constrained fits
decomps["NeII"] = d                                                # replace, then re-run Step 4
```

Useful checks when tuning a line: the spaxel fits (`plot_spaxel_fits`, `plot_decomposition_spaxels`), the residual
maps (`plot_residual_maps`), the kinematic residuals (`plot_kinematic_fit`), and how the stacked map changes when
the line is left out (`compare_stacks`).

### Running on another galaxy
Use **`02_template_new_galaxy.ipynb`**: the same workflow with no NGC 5728-specific settings. (If you copy this
notebook instead, change Section 0 (name, data folder, redshift); then the AGN positions (Section 1), the line
list and settings (Section 2, start from the defaults by emptying `LINE_SETTINGS`), `MODEL_PARAMS` and the line
groups (Section 4), and choose `DECOMP_PARAMS = "fit_mean"` or `"fit_per_line"` and `DECOMP_SIGMAS = "measured"`
(Section 5), and empty `AGN_OVERRIDES`, `FRACTION_SIGMA_OVERRIDES` and `DECOMP_RECUT`.) Saved analyses can be re-opened with `btb.Project.open("NAME")`; `btb.list_projects()` lists them.